# Group Isoforms into C-terminal and not C-terminal (Internal and N-terminal)
In the final database, we will add the entire new C-terminus for the isoforms that have a new C-terminus (can include several tryptic cleavage sites). For isoforms sharing a C-terminus with the canonical form, we will perform in-silico tryptic digestion and only add a unique fragment to the database (if we can find one).

# Setup
## Import and install Packages

In [ ]:
import sys
import os
import session_info

In [ ]:
import pandas as pd
import re

In [ ]:
# Display session information
session_info.show()

## Set working directory

In [ ]:
cwd = os.getcwd()
if not cwd.endswith("Isoform_Database/Jupyter_environment/Isoform_Database_SIAF/02_Isoform_Database"):
    print("WARNING: The working directory is not set to the '02_Isoform_Database'!")
    print("Current working directory:", cwd)
else:
    print(f"Working directory is correctly set to the '02_Isoform_Database' folder (\"{cwd}\").")

In [ ]:
# Data directories
mapping_dir = "../01_UniProt/data/mapping/"
unique_dir = "../01_UniProt/data/unique/"

## Read in fasta files and mapping

In [ ]:
full_proteome_unique = pd.read_csv(os.path.join(unique_dir, 'full_proteome_unique.csv'))

iso_canonical_mapping = pd.read_csv(os.path.join(mapping_dir, 'iso_canonical_mapping.csv'))
iso_canonical_mapping_flat = pd.read_csv(os.path.join(mapping_dir, 'iso_canonical_mapping_flat.csv'))

# Group Isoforms into C-terminal and not C-terminal

In [ ]:
# fast lookup dictionary from unique proteome dataframe
seq_lookup = full_proteome_unique.set_index('ID')['seq'].to_dict()

def process_isoforms(mapping_df, seq_dict, pattern):
    c_term_list = []
    others_list = []
    processed_isoforms = set()

    for _, row in mapping_df.iterrows():
        c_id = row['Entry']
        i_id = row['Isoform_ID']

        # Skip Isoform_ID already processed
        if i_id in processed_isoforms:
            continue
            
        c_seq, i_seq = seq_dict.get(c_id), seq_dict.get(i_id)
        
        if not c_seq or not i_seq: continue

        # 1. Find the first difference (N -> C)
        min_len = min(len(c_seq), len(i_seq))
        diff_idx = next((i for i in range(min_len) if c_seq[i] != i_seq[i]), min_len)
        
        # 2. Extract the tails from the divergence point to the end
        i_tail = i_seq[diff_idx:]
        c_tail = c_seq[diff_idx:]
        
        # 3. REFINED CHECK: 
        # It's only a C-term isoform if the END of the isoform 
        # is significantly different from the END of the canonical.
        # We check if the last 10 amino acids of the isoform exist 
        # at the end of the canonical sequence.
        
        anchor_length = 10 # Adjust based on how strict you want to be
        if len(i_tail) >= anchor_length:
            is_c_term = i_tail[-anchor_length:] != c_seq[-anchor_length:]
        else:
            is_c_term = i_tail != c_tail

        if is_c_term:
            # --- C-Terminal Extraction Logic ---
            upstream = i_seq[:diff_idx]
            cleavages = [m.start() for m in pattern.finditer(upstream)]
            start_pos = (cleavages[-1] + 1) if cleavages else 0
            pep = i_seq[start_pos:]
            
            c_term_list.append({
                'Isoform_ID': i_id,
                'Canonical_ID': c_id,
                'Type': 'C-terminal',
                'Peptide': pep,
                'Peptide_Length': len(pep)
            })
        else:
            # --- Internal/N-terminal Extraction Logic ---
            others_list.append({
                'Isoform_ID': i_id,
                'Canonical_ID': c_id,
                'Type': 'Internal/N-term'
            })

        # --- MARK AS PROCESSED ---
        processed_isoforms.add(i_id)
            
    return pd.DataFrame(c_term_list), pd.DataFrame(others_list)

# Execution
try_pattern = re.compile(r'[KR]')
tryP_pattern = re.compile(r'[KR](?!P)')

df_c_terminal_try, df_others_try = process_isoforms_refined(iso_canonical_mapping_flat, seq_lookup, try_pattern)
df_c_terminal_tryP, df_others_tryP = process_isoforms_refined(iso_canonical_mapping_flat, seq_lookup, tryP_pattern)

# View results
print(f"Detected {len(df_c_terminal_try)} C-terminal isoforms and {len(df_others_try)} other isoforms.")

In [ ]:
#Check for duplicates 
c_term_try_duplicates = df_c_terminal_try[df_c_terminal_try.duplicated(subset=['Peptide'], keep=False)]
c_term_tryP_duplicates = df_c_terminal_tryP[df_c_terminal_tryP.duplicated(subset=['Peptide'], keep=False)]

print(f"duplicate rows try: {len(c_term_try_duplicates)}")
print(f"duplicate rows tryP: {len(c_term_tryP_duplicates)}")

In [ ]:
c_term_try_duplicates

In [ ]:
# remove duplicates from c_terminal df
df_c_terminal_try_final = df_c_terminal_try[
    ~df_c_terminal_try['Peptide'].isin(c_term_try_duplicates['Peptide'].unique())
]

df_c_terminal_tryP_final = df_c_terminal_tryP[
    ~df_c_terminal_tryP['Peptide'].isin(c_term_tryP_duplicates['Peptide'].unique())
]

These duplicates will not be added to the database because they are not uniquely identifiable. All other new C-terminal peptides will be added to the canonical sequences so that the non C-terminal isoform petides can be compared to the canonical sequences and the new c-terminal sequences.

In [ ]:
df_c_terminal_try_final.to_csv("data/df_c_terminal_try.csv", index=False)
df_others_try.to_csv("data/df_others_try.csv", index=False)

df_c_terminal_tryP_final.to_csv("data/df_c_terminal_tryP.csv", index=False)
df_others_tryP.to_csv("data/df_others_tryP.csv", index=False)

In [ ]:
df_c_terminal_tryP_final